In [2]:
import torch
import os
import sys

sys.path.append(os.path.abspath('..'))

from FaultGenerators.WeightFaultInjector import WeightFaultInjector
from FaultGenerators.WeightFault import WeightFault

# 1. Crea un modello giocattolo con un singolo layer Conv2d
class ToyNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=2, out_channels=3, kernel_size=3)

net = ToyNet()

# 2. Crea l'injector
injector = WeightFaultInjector(net)

# 3. Salva il valore originale di un peso specifico, PRIMA di tutto
tensor_index = (0, 0, 0, 0)
original_value = float(net.state_dict()['conv1.weight'][tensor_index])
print(f'Valore originale: {original_value}')

# 4. Costruisci una lista di fault (anche solo 1, per iniziare)
faults = [
    WeightFault(layer_name='conv1', tensor_index=tensor_index, bit=30)
]

# 5. Inietta
injector.inject_multi_bit_flip(faults)
faulted_value = float(net.state_dict()['conv1.weight'][tensor_index])
print(f'Valore dopo il fault: {faulted_value}')

# 6. Verifica che sia CAMBIATO
assert faulted_value != original_value, "ERRORE: il valore non è cambiato!"

# 7. Ripristina
injector.restore_golden_multi()
restored_value = float(net.state_dict()['conv1.weight'][tensor_index])
print(f'Valore dopo il restore: {restored_value}')

# 8. Verifica che sia tornato ESATTAMENTE uguale all'originale
assert restored_value == original_value, "ERRORE: il restore non ha funzionato!"

print('✅ Test base superato')

Valore originale: -0.08055122941732407
Valore dopo il fault: -2.741016300451856e+37
Valore dopo il restore: -0.08055122941732407
✅ Test base superato


In [5]:
# Tre fault su tre pesi diversi, simultanei
faults_multi = [
    WeightFault(layer_name='conv1', tensor_index=(0, 0, 0, 0), bit=5),
    WeightFault(layer_name='conv1', tensor_index=(1, 0, 1, 1), bit=20),
    WeightFault(layer_name='conv1', tensor_index=(2, 1, 2, 2), bit=31),
]

# Salva gli originali
originals = {f.tensor_index: float(net.state_dict()['conv1.weight'][f.tensor_index])
             for f in faults_multi}

# Inietta tutti e tre insieme
injector.inject_multi_bit_flip(faults_multi)

print("\n--- VALORI DOPO INJECT (3 fault simultanei) ---")
for f in faults_multi:
    val = float(net.state_dict()['conv1.weight'][f.tensor_index])
    print(f"  {f.tensor_index} (bit {f.bit}): {val}   [originale era {originals[f.tensor_index]}]")

# Ripristina tutti
injector.restore_golden_multi()

# Verifica che TUTTI E TRE siano tornati esatti
for f in faults_multi:
    current = float(net.state_dict()['conv1.weight'][f.tensor_index])
    assert current == originals[f.tensor_index], f"ERRORE: {f.tensor_index} non è tornato originale"

print('✅ Test multi-fault superato')


--- VALORI DOPO INJECT (3 fault simultanei) ---
  (0, 0, 0, 0) (bit 5): -0.08055146783590317   [originale era -0.08055122941732407]
  (1, 0, 1, 1) (bit 20): 0.15177969634532928   [originale era 0.13615469634532928]
  (2, 1, 2, 2) (bit 31): 0.16088126599788666   [originale era -0.16088126599788666]
✅ Test multi-fault superato


In [4]:
same_index = (0, 0, 0, 1)
faults_collision = [
    WeightFault(layer_name='conv1', tensor_index=same_index, bit=10),
    WeightFault(layer_name='conv1', tensor_index=same_index, bit=20),  # stesso peso!
]

original = float(net.state_dict()['conv1.weight'][same_index])

injector.inject_multi_bit_flip(faults_collision)
# qui il valore finale riflette SOLO l'ultimo fault applicato (bit=20),
# perché il secondo fault parte dal valore già modificato dal primo

injector.restore_golden_multi()
restored = float(net.state_dict()['conv1.weight'][same_index])

assert restored == original, "ERRORE: collisione non gestita correttamente"
print('✅ Test collisione superato')

✅ Test collisione superato
